# 02 — Rule Engine + Thompson Sampling Path: Model Adaptif

**Riset acuan:**
- DIAMANTE Trial (JMIR 2024): RL meningkatkan step counts vs schedule statis
- Mixed-Effects Bandit (PMC 2024-25): contextual bandit > vanilla Thompson sampling

**Implementasi:**
- **Phase 1 (current)**: Rule 3-cabang berdasarkan `weekly_score`
- **Phase 2 (NEW)**: Thompson Sampling contextual bandit untuk fine-tune

**Phase 1 Rules:**
| Score | Strategy | Volume Multiplier |
|---|---|---|
| < 50% | REDUCE | 0.7 |
| 50-80% | MAINTAIN_SWAP | 1.0 |
| > 80% | INTENSIFY | 1.15 |

**Phase 2 (kapan switch):**
- Setelah user akumulasi ≥ 4 minggu data
- Setelah `daily_logs` punya ≥ 100 entries per user
- Warm-up dengan rule selama 2-3 minggu dulu

**Output:**
- `output/preprocessed/replanning_dataset.parquet` — untuk XGBoost training
- Thompson Sampling simulator (untuk validation)

In [11]:
import pandas as pd
import numpy as np
import json
import os
from scipy.stats import beta

np.random.seed(42)
os.makedirs('output/preprocessed', exist_ok=True)

df = pd.read_csv('output/eda/synthetic_with_score.csv')
print(f'Loaded: {df.shape}')

Loaded: (2773, 18)


## Phase 1: Rule 3-Cabang

In [12]:
INTENSITY_ORDER = ['LOW', 'MID', 'HIGH']

def classify_strategy(weekly_score: float) -> str:
    if weekly_score < 50:   return 'REDUCE'
    if weekly_score <= 80:  return 'MAINTAIN_SWAP'
    return 'INTENSIFY'

def get_volume_multiplier(strategy: str, user_choice: str = 'MODERATE') -> float:
    base = {'REDUCE': 0.7, 'MAINTAIN_SWAP': 1.0, 'INTENSIFY': 1.15}[strategy]
    if user_choice == 'AGGRESSIVE' and strategy != 'REDUCE':
        base = min(base * 1.10, 1.30)
    elif user_choice == 'KEEP':
        base = 1.0
    return round(base, 2)

def shift_intensity(current_intensity: str, strategy: str) -> str:
    idx = INTENSITY_ORDER.index(current_intensity) if current_intensity in INTENSITY_ORDER else 1
    if strategy == 'REDUCE':    idx = max(0, idx - 1)
    if strategy == 'INTENSIFY': idx = min(2, idx + 1)
    return INTENSITY_ORDER[idx]

# Test
test_cases = [
    (45, 'LOW',  'MODERATE'),
    (65, 'MID',  'MODERATE'),
    (88, 'MID',  'MODERATE'),
    (88, 'MID',  'AGGRESSIVE'),
    (30, 'HIGH', 'KEEP'),
]
print('=== Phase 1 Rule Engine ===')
for score, intensity, choice in test_cases:
    strat = classify_strategy(score)
    mult = get_volume_multiplier(strat, choice)
    new_int = shift_intensity(intensity, strat)
    print(f'  Score={score:3d} | {intensity} | {choice:10s} → {strat:15s} | Vol×{mult} | → {new_int}')

=== Phase 1 Rule Engine ===
  Score= 45 | LOW | MODERATE   → REDUCE          | Vol×0.7 | → LOW
  Score= 65 | MID | MODERATE   → MAINTAIN_SWAP   | Vol×1.0 | → MID
  Score= 88 | MID | MODERATE   → INTENSIFY       | Vol×1.15 | → HIGH
  Score= 88 | MID | AGGRESSIVE → INTENSIFY       | Vol×1.26 | → HIGH
  Score= 30 | HIGH | KEEP       → REDUCE          | Vol×1.0 | → MID


## Phase 2: Thompson Sampling Contextual Bandit (NEW)

Berdasarkan DIAMANTE Trial (JMIR 2024) dan Mixed-Effects Bandit (PMC 2024-25).

In [13]:
class ThompsonSamplingBandit:
    """Contextual bandit dengan 3 arms: REDUCE, MAINTAIN_SWAP, INTENSIFY.
    Beta-Bernoulli model — Beta(α, β) per arm."""
    
    def __init__(self, arms=['REDUCE', 'MAINTAIN_SWAP', 'INTENSIFY']):
        self.arms = arms
        # Uniform prior: Beta(1, 1) per arm
        self.params = {arm: {'alpha': 1.0, 'beta': 1.0} for arm in arms}
    
    def select_arm(self, context=None):
        """Sample dari posterior tiap arm, pilih yang highest."""
        samples = {arm: beta.rvs(p['alpha'], p['beta'])
                   for arm, p in self.params.items()}
        return max(samples, key=samples.get)
    
    def update(self, arm, reward):
        """Update Beta(α, β) berdasarkan reward (0-1)."""
        self.params[arm]['alpha'] += reward
        self.params[arm]['beta'] += (1 - reward)
    
    def get_arm_means(self):
        """Expected reward per arm (untuk inspection)."""
        return {arm: p['alpha'] / (p['alpha'] + p['beta'])
                for arm, p in self.params.items()}

# Reward formula (DIAMANTE pattern)
def compute_reward(compliance: float, weight_progress: float,
                   dropout: bool = False) -> float:
    alpha, beta_w, gamma = 0.5, 0.4, 0.3
    r = alpha * compliance + beta_w * weight_progress - gamma * float(dropout)
    return max(0, min(1, r))

print('Thompson Sampling Bandit class siap.')

Thompson Sampling Bandit class siap.


In [14]:
# ── Simulasi Bandit (untuk validasi) ──
bandit = ThompsonSamplingBandit()

# Simulate 100 episodes
true_arm_rewards = {'REDUCE': 0.5, 'MAINTAIN_SWAP': 0.7, 'INTENSIFY': 0.6}
history = []

for episode in range(100):
    arm = bandit.select_arm()
    # Simulate reward (Bernoulli dengan probability dari true reward)
    reward = 1 if np.random.random() < true_arm_rewards[arm] else 0
    bandit.update(arm, reward)
    history.append({'episode': episode, 'arm': arm, 'reward': reward})

history_df = pd.DataFrame(history)
print('Distribusi pilihan arm dalam 100 episodes:')
print(history_df['arm'].value_counts())
print('\nLearned arm means:')
print(bandit.get_arm_means())
print('\nTrue arm rewards:')
print(true_arm_rewards)
print('\n✅ Bandit converged toward MAINTAIN_SWAP (highest true reward)')

Distribusi pilihan arm dalam 100 episodes:
arm
MAINTAIN_SWAP    72
INTENSIFY        23
REDUCE            5
Name: count, dtype: int64

Learned arm means:
{'REDUCE': 0.42857142857142855, 'MAINTAIN_SWAP': 0.7297297297297297, 'INTENSIFY': 0.56}

True arm rewards:
{'REDUCE': 0.5, 'MAINTAIN_SWAP': 0.7, 'INTENSIFY': 0.6}

✅ Bandit converged toward MAINTAIN_SWAP (highest true reward)


## Generate Dataset untuk XGBoost Training (notebook 03)

In [16]:
# ── Generate continuous target multiplier untuk XGBoost ──
records = []
for _, row in df.iterrows():
    score = float(row['simulated_weekly_score'])
    weight_diff = float(row['simulated_weight_diff'])
    bmi = float(row['BMI'])
    exp_raw = row.get('Experience_Level', np.nan)
    exp_level = int(exp_raw) if pd.notna(exp_raw) else int(df['Experience_Level'].median())
    age_raw = row.get('Age', np.nan)
    age = int(age_raw) if pd.notna(age_raw) else int(df['Age'].median())
    freq_raw = row.get('Workout_Frequency (days/week)', np.nan)
    freq = int(freq_raw) if pd.notna(freq_raw) else int(df['Workout_Frequency (days/week)'].median())
    
    strat = classify_strategy(score)
    mult = get_volume_multiplier(strat)
    
    # Ground truth multiplier dengan small noise around rule
    noise = np.random.normal(0, 0.03)
    true_mult = max(0.6, min(1.3, mult + noise))
    
    records.append({
        'weekly_score': score,
        'weight_diff_kg': weight_diff,
        'bmi': bmi,
        'experience_level': exp_level,
        'age': age,
        'workout_frequency': freq,
        'strategy': strat,
        'rule_multiplier': mult,
        'target_multiplier': round(true_mult, 3),
    })

df_replan = pd.DataFrame(records)
print(f'Replanning dataset: {df_replan.shape}')
print('\nStrategy distribution:')
print(df_replan['strategy'].value_counts())
df_replan.head()

Replanning dataset: (2773, 9)

Strategy distribution:
strategy
REDUCE           1552
MAINTAIN_SWAP     887
INTENSIFY         334
Name: count, dtype: int64


,weekly_score,weight_diff_kg,bmi,experience_level,age,workout_frequency,strategy,rule_multiplier,target_multiplier
0,77.167142,-1.538895,30.20,3,56,4,MAINTAIN_SWAP,1.0,0.967
1,43.117357,-0.692923,32.00,2,46,4,REDUCE,0.7,0.658
2,61.476885,-0.527900,24.71,2,32,4,MAINTAIN_SWAP,1.0,1.040
3,40.230299,-0.504944,18.41,1,25,3,REDUCE,0.7,0.709
4,22.658466,-0.493198,14.39,1,38,3,REDUCE,0.7,0.667


In [17]:
df_replan.to_parquet('output/preprocessed/replanning_dataset.parquet', index=False)

# Save bandit state untuk reproducibility
bandit_state = bandit.params
with open('output/preprocessed/bandit_state.json', 'w') as f:
    json.dump(bandit_state, f, indent=2)

print('✅ Saved:')
print('  output/preprocessed/replanning_dataset.parquet')
print('  output/preprocessed/bandit_state.json')
print(f'\nNext: 03_training_evaluation.ipynb (XGBoost Regressor + bandit validation)')

✅ Saved:
  output/preprocessed/replanning_dataset.parquet
  output/preprocessed/bandit_state.json

Next: 03_training_evaluation.ipynb (XGBoost Regressor + bandit validation)
